#  House Price Prediction

we want to predict the market value of residential properties based on historical data.
The goal is to identify key features—like square footage, number of rooms, location, and so forth—that drive pricing, so that we can automate estimates for new listings.
Now, for the dataset, I assume you have a structured dataset with these features and corresponding prices.


## 1. Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

## 2. Loading the Dataset
 

In [ ]:
df = pd.read_csv("dataset/HousePricePrediction.csv")

df.head()

In [ ]:
df.info()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(x="SalePrice", y="LotArea", data=df)
plt.title("LotArea vs Sale Price")
plt.xlabel("Sale Price")
plt.ylabel("LotArea")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(df["OverallCond"],showmeans=True)
plt.title("Distribution of Sale Price")
plt.xlabel("Sale Price")
plt.ylabel("Count")
plt.show()

### Box Plot for Outlier Detection


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(y=df["SalePrice"])
plt.title("Sale Price Outliers")
plt.show()


### Total Basement Area vs Sale Price


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x=df["TotalBsmtSF"],
    y=df["SalePrice"],
    hue=df["OverallCond"]
)
plt.title("Basement Area vs Sale Price")
plt.xlabel("Total Basement SF")
plt.ylabel("Sale Price")
plt.show()

### Living Area vs Sale Price


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x=df["LotArea"],
    y=df["SalePrice"]
)
plt.title("Living Area vs Sale Price")
plt.xlabel("Ground Living Area (sq ft)")
plt.ylabel("Sale Price")
plt.show()

## 4. Data Cleaning

In [ ]:
isnull_counts = df.isnull().sum()

isnull_counts

In [ ]:
#dropping Irrelevant or High-NaN Columns
df = df.drop("Id", axis=1)

#dropping Rows with Missing Target Values
df = df.dropna(subset=["SalePrice"])

## 5. Feature and Target Separation

In [ ]:
category_var = df.select_dtypes(include = 'object')
num_var = df.select_dtypes(exclude = 'object')

print("Number of categorical features are: ", category_var.shape[1])
print("Number of numerical features are: ", num_var.shape[1])

In [ ]:
X = df.drop("SalePrice", axis=1)
Y = df["SalePrice"]

## 6. Feature Encoding


In [ ]:
#One-hot encode categorical features

cat_cols = X.select_dtypes(include=["object"]).columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

#Convert boolean columns to integers

bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

X


## 7. Train-Test Split


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42
)

## 8. Model Training and Evaluation


In [ ]:
#Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

lr_model = LinearRegression()
lr_model.fit(X_train, Y_train)

Y_pred_lr = lr_model.predict(X_test)
r2_lr = r2_score(Y_test, Y_pred_lr)

r2_lr

### Lasso Regression (L1 Regularization)


In [ ]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
lasso_model = Lasso(alpha=0.001)
lasso_model.fit(X_train_scaled, Y_train)

# Predict
Y_pred_lasso = lasso_model.predict(X_test_scaled)

# Evaluate
r2_lasso = r2_score(Y_test, Y_pred_lasso)

r2_lasso

### Ridge Regression (L2 Regularization)


In [ ]:
from sklearn.linear_model import Ridge


ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, Y_train)

Y_pred_ridge = ridge_model.predict(X_test_scaled)
r2_ridge = r2_score(Y_test, Y_pred_ridge)

r2_ridge

### RandomForest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200)
rf.fit(X_train_scaled, Y_train)

Y_pred_rf = rf.predict(X_test_scaled)

r2_rf = r2_score(Y_test, Y_pred_rf)
r2_rf

### XGB Regression

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=300, learning_rate=0.05)
xgb.fit(X_train_scaled, Y_train)
Y_pred_xgb = xgb.predict(X_test_scaled)
r2_xgb = r2_score(Y_test, Y_pred_xgb)
r2_xgb

## 9. Model Performance Comparison

In [ ]:
#R² Score Comparison Table

results_df = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression", "Lasso Regression", "Random Forest", "XGBoost"],
    "R2 Score": [r2_lr, r2_ridge, r2_lasso, r2_rf, r2_xgb]
})

results_df

#R² Score Comparison Visualization

plt.figure(figsize=(8, 5))
sns.barplot(
    x="Model",
    y="R2 Score",
    data=results_df
)
plt.title("Model Comparison using R² Score")
plt.ylabel("R² Score")
plt.ylim(0, 1)
plt.show()

# 10. Saving model 

In [ ]:
import pickle

# Save the baseline trained model to a file
with open("models/lr_model.pkl", "wb") as file:  #
    pickle.dump(lr_model, file)


# Save the Ridge trained model to a file
with open("models/ridge_model.pkl", "wb") as file:  #
    pickle.dump(ridge_model, file)

# Save the lasso trained model to a file
with open("models/lasso_model.pkl", "wb") as file:  #
    pickle.dump(lasso_model, file)

# 11. Saving the Package Version

In [ ]:
import pkg_resources

# List of the packages you know you're using
required_packages = [
    'numpy',
    'pandas',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'ipykernel',
]

requirements = []

for package in required_packages:
    try:
        version = pkg_resources.get_distribution(package).version
        requirements.append(f"{package}=={version}")
    except pkg_resources.DistributionNotFound:
        print(f"Package {package} not found in the environment.")

#requirements to a file
with open('requirements.txt', 'w') as f:
    for line in requirements:
        f.write(line + '\n')